In [0]:
--Créer la dim_product :
--Ecrivez le DDL et chargez la dimension
--produit (dim_product) basé sur les
--tables product, product_model. Liez la
--fact_sales avec la dim_product.

USE CATALOG barbara_lakehouse;
USE SCHEMA gold;

DECLARE OR REPLACE load_date = current_timestamp();
VALUES load_date;

CREATE TABLE IF NOT EXISTS barbara_lakehouse.gold.dim_product (
  _tf_dim_product_id      BIGINT GENERATED ALWAYS AS IDENTITY,
  -- Business key
  prod_product_id         INT,
  -- Attributs produit
  prod_name               STRING,
  prod_product_number     STRING,
  prod_color              STRING,
  prod_standard_cost      DECIMAL(19,4),
  prod_list_price         DECIMAL(19,4),
  prod_size               STRING,
  prod_weight             DECIMAL(8,2),
  -- Modèle produit
  prod_product_model_id   INT,
  prod_model_name         STRING,
  -- Dates produit
  prod_sell_start_date    TIMESTAMP,
  prod_sell_end_date      TIMESTAMP,
  prod_discontinued_date  TIMESTAMP,
  -- SCD2 metadata
  _tf_valid_from          TIMESTAMP,
  _tf_valid_to            TIMESTAMP,
  _tf_create_date         TIMESTAMP,
  _tf_update_date         TIMESTAMP
)
USING DELTA;

CREATE OR REPLACE TEMP VIEW _tmp_dim_product AS
SELECT
  p.product_id                    AS prod_product_id,
  p.name                          AS prod_name,
  p.product_number                AS prod_product_number,
  p.color                         AS prod_color,
  p.standard_cost                 AS prod_standard_cost,
  p.list_price                    AS prod_list_price,
  p.size                          AS prod_size,
  p.weight                        AS prod_weight,
  p.product_model_id              AS prod_product_model_id,
  pm.name                         AS prod_model_name,
  p.sell_start_date               AS prod_sell_start_date,
  p.sell_end_date                 AS prod_sell_end_date,
  p.discontinued_date             AS prod_discontinued_date
FROM barbara_lakehouse.silver.product p
LEFT JOIN barbara_lakehouse.silver.productmodel pm
  ON p.product_model_id = pm.product_model_id
 AND pm._tf_valid_to IS NULL
WHERE p._tf_valid_to IS NULL;

MERGE INTO barbara_lakehouse.gold.dim_product AS tgt
USING _tmp_dim_product AS src
ON tgt.prod_product_id = src.prod_product_id
AND tgt._tf_valid_to IS NULL
WHEN MATCHED AND (
       tgt.prod_name              IS DISTINCT FROM src.prod_name
    OR tgt.prod_product_number    IS DISTINCT FROM src.prod_product_number
    OR tgt.prod_color             IS DISTINCT FROM src.prod_color
    OR tgt.prod_standard_cost     IS DISTINCT FROM src.prod_standard_cost
    OR tgt.prod_list_price        IS DISTINCT FROM src.prod_list_price
    OR tgt.prod_size              IS DISTINCT FROM src.prod_size
    OR tgt.prod_weight            IS DISTINCT FROM src.prod_weight
    OR tgt.prod_product_model_id  IS DISTINCT FROM src.prod_product_model_id
    OR tgt.prod_model_name        IS DISTINCT FROM src.prod_model_name
    OR tgt.prod_sell_start_date   IS DISTINCT FROM src.prod_sell_start_date
    OR tgt.prod_sell_end_date     IS DISTINCT FROM src.prod_sell_end_date
    OR tgt.prod_discontinued_date IS DISTINCT FROM src.prod_discontinued_date
) THEN
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date

WHEN NOT MATCHED BY SOURCE
AND tgt._tf_valid_to IS NULL THEN
  UPDATE SET
    tgt._tf_valid_to    = load_date,
    tgt._tf_update_date = load_date
;

MERGE INTO barbara_lakehouse.gold.dim_product AS tgt
USING _tmp_dim_product AS src
ON tgt.prod_product_id = src.prod_product_id
AND tgt._tf_valid_to IS NULL
WHEN NOT MATCHED THEN
  INSERT (
    prod_product_id,
    prod_name,
    prod_product_number,
    prod_color,
    prod_standard_cost,
    prod_list_price,
    prod_size,
    prod_weight,
    prod_product_model_id,
    prod_model_name,
    prod_sell_start_date,
    prod_sell_end_date,
    prod_discontinued_date,
    _tf_valid_from,
    _tf_valid_to,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.prod_product_id,
    src.prod_name,
    src.prod_product_number,
    src.prod_color,
    src.prod_standard_cost,
    src.prod_list_price,
    src.prod_size,
    src.prod_weight,
    src.prod_product_model_id,
    src.prod_model_name,
    src.prod_sell_start_date,
    src.prod_sell_end_date,
    src.prod_discontinued_date,
    load_date,
    NULL,
    load_date,
    load_date
  )
;

ALTER TABLE barbara_lakehouse.gold.fact_sales
ADD COLUMN product_id INT;

ALTER TABLE barbara_lakehouse.gold.fact_sales
ADD COLUMN _tf_dim_product_id BIGINT;


CREATE OR REPLACE TEMP VIEW _tmp_fact_sales AS
SELECT
    CAST(soh.sales_order_id AS INT) AS sales_order_id,
    CAST(sod.sales_order_detail_id AS INT) AS sales_order_detail_id,
    -- NEW: product_id
    CAST(sod.product_id AS INT) AS product_id,
    -- calendar
    10000 * YEAR(soh.order_date) + 100 * MONTH(soh.order_date) + DAY(soh.order_date) AS _tf_dim_calendar_id,
    COALESCE(cust._tf_dim_customer_id, -9) AS _tf_dim_customer_id,
    COALESCE(geo._tf_dim_geography_id, -9) AS _tf_dim_geography_id,
    -- measures
    COALESCE(CAST(sod.order_qty AS SMALLINT), 0) AS sales_order_qty,
    COALESCE(CAST(sod.unit_price AS DECIMAL(19,4)), 0) AS sales_unit_price,
    COALESCE(CAST(sod.unit_price_discount AS DECIMAL(19,4)), 0) AS sales_unit_price_discount,
    COALESCE(CAST(sod.line_total AS DECIMAL(38, 6)), 0) AS sales_line_total
FROM barbara_lakehouse.silver.sales_order_detail sod
LEFT JOIN barbara_lakehouse.silver.sales_order_header soh
  ON sod.sales_order_id = soh.sales_order_id AND soh._tf_valid_to IS NULL
LEFT JOIN barbara_lakehouse.silver.customer c
  ON soh.customer_id = c.customer_id AND c._tf_valid_to IS NULL
LEFT JOIN barbara_lakehouse.gold.dim_customer cust
  ON c.customer_id = cust.cust_customer_id
LEFT JOIN barbara_lakehouse.silver.address a
  ON soh.bill_to_address_id = a.address_id AND a._tf_valid_to IS NULL
LEFT JOIN barbara_lakehouse.gold.dim_geography geo
  ON a.address_id = geo.geo_address_id
WHERE sod._tf_valid_to IS NULL;

MERGE INTO barbara_lakehouse.gold.fact_sales AS tgt
USING _tmp_fact_sales AS src
ON tgt.sales_order_detail_id = src.sales_order_detail_id
AND tgt.sales_order_id = src.sales_order_id
WHEN MATCHED AND (
       tgt.product_id              IS DISTINCT FROM src.product_id
    OR tgt._tf_dim_calendar_id     IS DISTINCT FROM src._tf_dim_calendar_id
    OR tgt._tf_dim_customer_id     IS DISTINCT FROM src._tf_dim_customer_id
    OR tgt._tf_dim_geography_id    IS DISTINCT FROM src._tf_dim_geography_id
    OR tgt.sales_order_qty         IS DISTINCT FROM src.sales_order_qty
    OR tgt.sales_unit_price        IS DISTINCT FROM src.sales_unit_price
    OR tgt.sales_unit_price_discount IS DISTINCT FROM src.sales_unit_price_discount
    OR tgt.sales_line_total        IS DISTINCT FROM src.sales_line_total
) THEN
  UPDATE SET
    tgt.product_id              = src.product_id,
    tgt._tf_dim_calendar_id     = src._tf_dim_calendar_id,
    tgt._tf_dim_customer_id     = src._tf_dim_customer_id,
    tgt._tf_dim_geography_id    = src._tf_dim_geography_id,
    tgt.sales_order_qty         = src.sales_order_qty,
    tgt.sales_unit_price        = src.sales_unit_price,
    tgt.sales_unit_price_discount = src.sales_unit_price_discount,
    tgt.sales_line_total        = src.sales_line_total,
    tgt._tf_update_date         = load_date
WHEN NOT MATCHED THEN
  INSERT (
    sales_order_id,
    sales_order_detail_id,
    product_id,
    _tf_dim_calendar_id,
    _tf_dim_customer_id,
    _tf_dim_geography_id,
    sales_order_qty,
    sales_unit_price,
    sales_unit_price_discount,
    sales_line_total,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.sales_order_id,
    src.sales_order_detail_id,
    src.product_id,
    src._tf_dim_calendar_id,
    src._tf_dim_customer_id,
    src._tf_dim_geography_id,
    src.sales_order_qty,
    src.sales_unit_price,
    src.sales_unit_price_discount,
    src.sales_line_total,
    load_date,
    load_date
  );

MERGE INTO barbara_lakehouse.gold.fact_sales AS f
USING (
  SELECT
    prod_product_id,
    _tf_dim_product_id
  FROM barbara_lakehouse.gold.dim_product
  WHERE _tf_valid_to IS NULL
) AS dp
ON f.product_id = dp.prod_product_id
WHEN MATCHED THEN
  UPDATE SET
    f._tf_dim_product_id = COALESCE(dp._tf_dim_product_id, -9);

SELECT
  COUNT(*) AS nb_rows,
  SUM(CASE WHEN _tf_dim_product_id IS NULL OR _tf_dim_product_id = -9 THEN 1 ELSE 0 END) AS nb_unknown_product
FROM barbara_lakehouse.gold.fact_sales;
